# പാഠം 18: ക്രിപ്റ്റോഗ്രാഫിക് റസീറ്റുകളിലൂടെ AI ഏജന്റുകൾ സുരക്ഷിതമാക്കൽ

## ഹാൻഡ്‌സ്ഒൺ നോട്ട്‌ബുക്ക്

ഈ നോട്ട്‌ബുക്ക് നാലു പാഠങ്ങൾ വഴിപാടായി നടത്തുന്നു:

1. ഏജന്റ് ടൂൾ കോൾ ഒരു ലേഖനത്തിന്റെ **നിങ്ങളുടെ ആദ്യ റസീറ്റ് ഒപ്പിടുക** കൂടാതെ അതിനെ സ്ഥിരീകരിക്കുക.
2. **റസീറ്റ് മോഷ്ടിക്കുക** കൂടാതെ സ്ഥിരീകരണം പരാജയപ്പെടുന്നത് കാണുക.
3. **മൂന്നു റസീറ്റ് ചെയിൻ നിർമ്മിക്കുക** കൂടാതെ ചെയിൻ സുതാര്യത ഉറപ്പാക്കുക.
4. **Microsoft Agent Framework ടൂൾ കോൾ റാപ്പ് ചെയ്യുക** എങ്ങനെ ഓരോ പ്രവർത്തനവും ഒരു റസീറ്റ് പുറത്ത് വിടുന്നു.

എല്ലാ ക്രിപ്റ്റോഗ്രാഫിക് പ്രിമിറ്റീവുകളും നന്നായി പരിപാലിച്ച ലൈബ്രറികളിൽ നിന്നും ഇറക്കുമതി ചെയ്‌തതാണ് (`pynacl` Ed25519-നായി, `jcs` RFC 8785 കാനോണിക്കല് JSON-നായി, Python സ്റ്റാൻഡേർഡ് ലൈബ്രറിയിലുള്ള `hashlib` SHA-256-നായി). റസീറ്റ് ലాజിക് സ്വയം സരളമായ Python ആണെന്നും വായിച്ചും മാറ്റുകയും ചെയ്യാവുന്നതുമാണ്.

കോശങ്ങൾ അനുക്രമമനുസരി പ്രവർത്തിപ്പിക്കുക. ഓരോ വകുപ്പ് ചെറിയതും സ്വയം സമാഹൃതവുമായതാണ്.


## സെറ്റ് അപ്പ്

രണ്ട് ഡിപണ്ടൻസികളും ഇൻസ്റ്റാൾ ചെയ്യുക. ഇരുവരുടെയും അനുമതി അനുകൂലമായ ലൈസൻസുകളാണ് (Apache-2.0 / MIT).


In [ ]:
!pip install -q pynacl jcs

In [ ]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import BadSignatureError
from jcs import canonicalize

## സഹായക ഉപകരണങ്ങൾ

ഈ രണ്ട് സഹായികൾ പാഡിങ് ഇല്ലാതെ base64url എൻകോഡിംഗും ഏതെങ്കിലും ഒബ്ജക്റ്റുകളുടെ SHA-256 ഹാഷിംഗും കൈകാര്യം ചെയ്യുന്നു. അവയുടെ കാര്യം നോട്ട്ബുക്കിന്റെ ബാക്കി ഭാഗം റിസീറ്റ് ലജിക്ക് തന്നെ ശ്രദ്ധിക്കുമ്പോൾ നിർത്തിപിടിക്കുന്നു.


In [ ]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """Decode a base64url string that may be missing padding."""
    padding = "=" * ((4 - len(s) % 4) % 4)
    return base64.urlsafe_b64decode(s + padding)

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Section 1: നിങ്ങളുടെ ആദ്യ രസീത് ഒപ്പുവയ്ക്കുക

**Contoso Travel** എന്ന ഞങ്ങളുടെ ഏജന്റ് സിഡ്‌നിയിൽ നിന്നും ലോസ് എഞ്ചലസിലേക്ക് ഒരു ഉപഭോക്താവിനായി ഫ്‌ളൈറ്റുകൾ തിരഞ്ഞെടുത്തതായി تصور ചെയ്യുക. ഒരു ഭാവി ഓഡിറ്റർ ഞങ്ങളെ വിശ്വസിക്കാതെ തന്നെ അത് പരിശോധിക്കാൻ ഈ കാൾ ഒപ്പിട്ട ഒരു രസീത് ആയി രേഖപ്പെടുത്തണം.

### Step 1.1: ഒപ്പുവയ്ക്കലിനുള്ള കീ സൃഷ്ടിക്കുക

പ്രോഡക്ഷനിൽ, ഏജന്റിന്റെ ഒപ്പിടൽ കീ ഹാർഡ്‌വെയർ സെക്യൂരിറ്റി മോട്യൂളിൽ (HSM), അസ്യൂർ കീ വോൾട്ട്, അല്ലെങ്കിൽ സമാനമായ ഒരു സംരക്ഷിത സംഭരണയിലുണ്ടാകും. ഈ പാഠത്തിനായി നാം മെമ്മറിയിൽ ഒരു പുതിയ കീ സൃഷ്ടിക്കുന്നു.


In [ ]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

### Step 1.2: reçെറ്റ് പെയ്‌ലോഡ് നിർമ്മിക്കുക

പെയ്‌ലോഡ് reçെറ്റ് സ്‌നിഷ്ടപ്പെടുത്താൻ വേണ്ടതെല്ലാം ഉൾക്കൊള്ളുന്നു: ആരാണ് പ്രവർത്തിച്ചത്, ഏത് ടൂൾ, ഏത് പാരാമീറ്ററുകളോടെ, എന്താണ് തിരികെ വന്നത്, ഏതു പോളിസിയുടെ കീഴിൽ, കൂടാതെ ഏപ്പോൾ. പാരാമീറ്ററുകളും ഫലവും.inline ആയി ഉൾപ്പെടുത്താതിരിക്കേണ്ടത് reçെറ്റ് സംവേദനാത്മക ഉള്ളടക്കം പുറത്ത് വിടാതിരിക്കുവാനുള്ളതാണ്, അതിനാൽ അവയുടെ ഹാഷിങ്ങാണ് ചെയ്യുന്നത്.


In [ ]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

### Step 1.3: സമ്മതപ്പെടുത്തിയശ്ശ് ഒപ്പിട്ട് സ്വീകരണ പത്രം_ready ചെയ്യുക

മൂന്ന് ഘട്ടങ്ങൾ:

1. JCS ഉപയോഗിച്ച് പayload ന് കാനോണിക്കൽ രൂപമാക്കുക, അതിലൂടെ ഇതേ താത്വിക സ്വീകരണപത്രം സൃഷ്ടിക്കുന്ന രണ്ട് നടപ്പാക്കലുകളും ബൈറ്റ്-ഒരുപോലെ ആയ ബൈറ്റുകൾ ഉല്പാദിപ്പിക്കും.
2. കാനോണിക്കൽ ബൈറ്റുകൾ SHA-256 ഉപയോഗിച്ച് ഹാഷ് ചെയ്യുക.
3. Ed25519 സ്വകാര്യ കീ ഉപയോഗിച്ച് ഹാഷിനെ ഒപ്പിടുക.

ഒപ്പിട്ട സ്വাক্ষരം തുടർന്ന് അമൂല്യ സ്വീകരണപത്രം നിർമ്മിക്കാൻ മൗലിക പayload-ലോട് ചേർക്കപ്പെടുന്നു.


In [ ]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()
    signature_bytes = signing_key.sign(message_hash).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

### ഘട്ടം 1.4: رسید് പരിശോധിക്കുക

പരിശോധനം പ്രക്രിയയെ മറിച്ചടിക്കുന്നു. ഞങ്ങൾ സിഗ്നേച്ചർ നീക്കം ചെയ്ത്, കാനോണിക്കൽ ഹാഷ് പുനർകണക്കാക്കി, رسیدിലെ പബ്ലിക് കീയോട് സിഗ്നേച്ചർ പരിശോധിക്കുന്നു.

ഈ പരിശോധന നടത്തുന്ന ഓഡിറ്റർക്ക് നമ്മിൽ നിന്നേക്കും വേണ്ടത് رسید് മാത്രമാണുള്ളത്. വിളിക്കേണ്ട സർവീസ് ഒന്നുമില്ല, ചോദിക്കേണ്ട കീ ഡയറക്ടറി ഒന്നുമില്ല, വിശ്വാസം ആവശ്യമുമില്ല.


In [ ]:
def verify_receipt(receipt: dict) -> bool:
    """
    Verify a receipt's Ed25519 signature.
    Returns True if valid, False otherwise.
    """
    sig_obj = receipt.get("signature")
    if not sig_obj or sig_obj.get("alg") != "EdDSA":
        return False

    # Reconstruct the payload that was actually signed (everything except signature).
    payload = {k: v for k, v in receipt.items() if k != "signature"}

    canonical = canonicalize(payload)
    message_hash = hashlib.sha256(canonical).digest()

    try:
        verify_key = signing.VerifyKey(b64url_decode(sig_obj["public_key"]))
        verify_key.verify(message_hash, b64url_decode(sig_obj["sig"]))
        return True
    except BadSignatureError:
        return False
    except Exception as exc:
        print(f"Verification error: {exc}")
        return False

is_valid = verify_receipt(receipt)
print(f"Receipt is valid: {is_valid}")

നിങ്ങൾക്ക് `Receipt is valid: True` കാണണം. ഏജന്റ് ആദ്യ ക്രിപ્ટോഗ്രാഫിക്കായി ഒപ്പിട്ട ഓഡിറ്റ് രേഖ തയ്യാറാകും.


## Section 2: ലഭ്യത്തിൽ അഴിമതി വരുത്തുക

ലഭ്യങ്ങളുടെ മുഴുവൻ ലക്ഷ്യം അവ അഴിമതി തെളിയിക്കുന്നതായുള്ളതാണ്. അത് നമുക്കു തെളിയിക്കാം.

നാം ഉപരിതലത്തിൽ ഒരു കൃത്യമായൊരു അക്ഷരം മാറ്റി സ്ഥിരീകരണം പരാജയപ്പെടുന്നത് കാണാം.


In [ ]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered)}")

### എന്ത് സംഭവിച്ചു?

`policy_id` മാറ്റിയപ്പോൾ, canonical bytes മാറി. അവയുടെ SHA-256 ഹാഷ് മാറി. ഒറിജിനൽ ഹാഷ് മേൽ ആയിരുന്ന ഒപ്പ് പുതിയ ഹാഷിനൊപ്പം artık പൊരുത്തപെടുന്നില്ല. സ്ഥിരീകരണം ശരിയായി `False` മടക്കിയെത്തിക്കുന്നു.

റിസീപ്പ്റ്റിന്റെ ഏത് ഫീൽഡും മാറ്റി അത് സ്ഥിരീകരിക്കാൻ സാധിക്കില്ല, മുമ്പോറുകാർക്ക് പ്രൈവറ്റ് കീ ഉണ്ടായിരിക്കാത്തവರೆಗೆ. പ്രൈവറ്റ് കീ ഒരു കീ വോൾട്ടിൽ ഉണ്ടാകുമ്പോഴും, പബ്ലിക്ക് കീ പ്രസിദ്ധീകരിച്ചിട്ടുണ്ടെങ്കിൽ, തട്ടിപ്പുകൾ മറയ്ക്കാൻ കഴിയില്ല.

സ്വയം പരീക്ഷിക്കുക: മുകളിൽ സെല്ലിൽ `tool_name` അല്ലെങ്കിൽ `agent_id` അല്ലെങ്കിൽ `timestamp` മാറ്റി വീണ്ടും പ്രവർത്തിപ്പിക്കൂ. ഓരോ മാറ്റവും അസാധുവായ റിസീപ്പ്റ്റ് സൃഷ്‌ടിക്കും.


## Section 3: റെസിപ്പ്റ്റുകൾ ചെയിൻ പോലെ ബന്ധിപ്പിക്കുക

ഒരു സിംഗിൾ റെസിപ്പ്റ്റ് ഒരു ആക്ഷൻ മാത്രം സംരക്ഷിക്കുന്നു. മിക്ക ഏജന്റുകൾ വിവിധ ആക്ഷനുകൾ ചെയ്യുന്നു. മുഴുവൻ സീക്വൻസിനെ ടെമ്പർ-എവിഡന്റ് ചെയ്യാനായി, ഓരോ റെസിപ്പ്റ്റും മുൻറെസിപ്പ്റ്റിന്റെ ഹാഷ് പുതിയ റെസിപ്പ്റ്റിന്റെ പേലോഡിൽ ഉൾപ്പെടുത്തി മുൻറെസിപ്പ്റ്റുമായി ബന്ധിപ്പിക്കുന്നു.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

ഏവർക്കും ഒരു റെസിപ്പ്റ്റ് ഒഴിവാക്കുകയോ ക്രമം മാറ്റുകയോ ചെയ്താൽ, ചെയിൻ ആ പോയിന്റിൽ തന്നെ തകർക്കപ്പെടും. ഏതൊരു പിന്നീട് വരുന്ന റെസിപ്പ്റ്റിന്റെയും സാധുത പരാജയപ്പെടും, കാരണം അതിന്റെ `previous_receipt_hash` ഇതിനു മുമ്പുള്ള റെസിപ്പ്റ്റിന്റെ യഥാർത്ഥ ഹാഷിനെ മാപ്പ് ചെയ്യുന്നതല്ല.


In [ ]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [ ]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

In [ ]:
def verify_chain(chain: list) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt (except the genesis) must reference the previous receipt's hash.
      3. Sequence numbers must match each receipt's zero-based position in the chain.
    Returns a list of per-receipt result dicts.
    """
    results = []
    for i, receipt in enumerate(chain):
        sig_ok = verify_receipt(receipt)

        if i == 0:
            chain_ok = receipt["previous_receipt_hash"] is None
        else:
            expected = receipt_hash(chain[i - 1])
            chain_ok = receipt["previous_receipt_hash"] == expected

        seq_ok = receipt["sequence"] == i

        results.append({
            "index": i,
            "tool": receipt["tool_name"],
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok,
        })
    return results

for r in verify_chain(chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

ഇപ്പോൾ മദ്ധ്യത്തിലിരിക്കുന്ന രസീത് മാറിമാറ്റം ചെയ്ത് ശൃംഖല തകര്ക്കുക, പിന്നെ വീണ്ടും പരിശോധന നടത്തുക. മാറിയ രസീത് അതിന്റെ ഒപ്പ് പരിശോധനയിൽ പരാജയപ്പെടും, കൂടാതെ അടുത്ത രസീത് അതിന്റെ ശൃംഖല-ജോഡി പരിശോധനയിൽ പരാജയപ്പെടും (കാരണം അതിന്റെ `previous_receipt_hash` ഇനി മാറ്റിയ മദ്ധ്യ രസീതിന്റെ ഹാഷിനോട് പൊരുത്തപ്പെടുന്നില്ല).


In [ ]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 ഇപ്പോഴും പരിശോദന നടത്തുന്നുണ്ട് (അത് സാരമായില്ല, കൂടാതെ ആശ്രയിക്കാൻ മുമ്പത്തെ ഒരിടവശവും ഇല്ല). Receipt 1 അതിന്റെ സിഗ്നേച്ചർ പരിശോധനയിൽ പരാജയപ്പെടുന്നു കാരണം ഞങ്ങൾ `tool_args_hash` മാറ്റി. Receipt 2 അതിന്റെ ചൈൻ-ലിങ്ക് പരിശോധനയിൽ പരാജയപ്പെടുന്നു കാരണം അതിന്റെ `previous_receipt_hash` യഥാർത്ഥ (ഇപ്പോൾ മാറ്റപ്പെട്ട) receipt 1-നെ പ്രതിമാസിച്ച് കണക്കാക്കിയിരിക്കുന്നു.

മാറ്റം ചെയ്ത receipt 1 വീണ്ടും മറ്റൊരാൾ സൈൻ ചെയ്‌താലും (അവർ സ്വകാര്യ കീയെ കൂടാതെ അത് സാധ്യമല്ല), receipt 2-ൽ ചൈൻ-ലിങ്ക് വ്യത്യാസം ഇപ്പോഴും തട്ടിപ്പ് തുറന്നുപറയും. മാറ്റം മറയ്ക്കാനായി, ആക്രമകൻ മാറ്റം തുടങ്ങുന്നreceipt-ൽ നിന്ന് എല്ലാ receipts-ഉം വീണ്ടും സൈൻ ചെയ്യേണ്ടി വരും, അത് സ്വകാര്യ കീയുടെ ഉടമസ്ഥത ആവശ്യമാണ്.


## വിഭാഗം 4: എജന്റ് ടൂൾ കോൾ ഒരു റസീറ്റ് ഒപ്പ് ചേർത്ത് റാപ്പർ ചെയ്യുക

ഒരു യഥാർത്ഥ പ്രയോഗത്തിൽ, ഓരോ എജന്റ് റൈറ്ററിനും `make_receipt` വിളിക്കാൻ ഓർക്കണമെന്ന് നിങ്ങൾ ആഗ്രഹിക്കുന്നില്ല. ഓരോ ടൂൾ വിളിപ്പാടിനും റസീറ്റ് ഒപ്പ് ചേർക്കൽ സ്വയംസ്ഫೂರ്തിയാക്കാൻ നിങ്ങൾ ആഗ്രഹിക്കുന്നു.

ഇവിടെ ഏറ്റവും ലളിതമായ മാതൃകയുണ്ട്: ഒരു റാപ്പർ ക്ലാസ്, ഇത് ഏതെങ്കിലും കോളബിള്‍ ടൂൾ ഫങ്ഷൻ എടുത്ത് അതിന്റെ റസീറ്റ്-ഇഷ്യൂ ചെയ്യുന്ന പതിപ്പായി തിരികെ നൽകുന്നു. ഇത് മൈക്രോസോഫ്റ്റ് എജന്റ് ഫ്രെയിംവർക്ക് (`agent_framework.azure`) ഉൾപ്പെടെ ഏതൊരു എജന്റ് ഫ്രെയിംവർക്കിനും അനുയോജ്യമാക്കുന്നു.

നിങ്ങൾക്ക് Azure AI Foundry പ്രോജക്റ്റ് സജ്ജമാക്കിയിട്ടില്ലെങ്കിൽ പോലും, താഴെയുള്ള ലോക്കൽ മോക് ഈ മാതൃകം കാണിക്കുന്നു.


In [ ]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [ ]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to AzureAIProjectAgentProvider as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")

### Microsoft ഏജന്റ് ഫ്രെയിമ്വർക്കുമായി സംയോജിപ്പിക്കൽ

മുകളിൽ കൊടുത്തിരിക്കുന്ന `ReceiptedTool` റാപ്പർ ഫ്രെയിംവർക്കിന്റെ പരിധിയിലല്ല. ഇത് Microsoft ഏജന്റ് ഫ്രെയിമ്വർക്കിൽ നിർമ്മിച്ച ഏജന്റ്‌ന്റെ ഉള്ളിൽ ഉപയോഗിക്കാൻ, റാപ്പുചെയ്ത ഫംഗ്ഷനിനെ ടൂളായി രജിസ്റ്റർ ചെയ്യണം. ഒരു סכീച്ച് (നിങ്ങൾ യഥാർത്ഥ Azure AI Foundry ടൂൾ രജിസ്‌ട്രേഷനുമായി മോക്ക് മാറ്റം വരുത്തും):

```python
# സംയോജന രൂപം കാണിക്കുന്ന સંદાન કોડ.
# from agent_framework.azure import AzureAIProjectAgentProvider
# from azure.identity import AzureCliCredential
#
# provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())
# agent = provider.create_agent(
#     നിർദേശങ്ങൾ="നീ ഒരു കോൺടോസോ ട്രാവൽ ഏജന്റ് ആണ് ...",
#     tools=[receipted_lookup],   # ആक्रमിച്ച ടൂൾ, മൂടാത്ത ഫംഗ്ഷൻ അല്ല
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # റൺ കഴിഞ്ഞ്, ഏജന്റ് നടത്തിയ എല്ലാ ടൂൾ കോൾക്കും ഒപ്പിട്ട രസീത് ഉണ്ട്:
# audit_chain = receipted_lookup.receipts
```

ഏജന്റ് ഫ്രെയിംവർക്ക് റെസിപ്പ്റ്റുകൾക്ക് കാര്യമില്ല. റെസിപ്പ്റ്റ് സൈൻ ചെയ്യൽ ടൂളിന് ചുറ്റും റാപ്പ് ചെയ്തിരിക്കുന്നു, ഫ്രെയിംവർക്കിൽ ഒരു ഘടകമായി ചേർത്തിട്ടില്ല. ഏജന്റ് പരിഷ്കരിക്കാതെ നിലവിലെ ഏജന്റ് കോഡിൽ പ്രോവനെൻസ് ചേർക്കാനുള്ള മാർഗമാണ് ഇത്.


## ഒർമ്മപ്പെടുത്തലും വിപുലീകരണ ചലഞ്ചും

നിങ്ങൾ ചെയ്തു കഴിഞ്ഞത്:

- Ed25519 കീ ജോഡി സൃഷ്ടിച്ചു.
- ഏജന്റ് ടൂൾ കോൾക്കായി ഒരു റെസീറ്റ് കെട്ടിപ്പുഴുത്തു, ഒപ്പിട്ടു.
- മാത്രം പബ്ലിക് കീ ഉപയോഗിച്ച് റെസീറ്റ് ഓഫ്‌ലൈൻ സ്ഥിരീകരിച്ചു.
- ഒരു റെസീറ്റ് തട്ടിയെടുത്തു, പരിശോധന പരാജയമായി എന്നത് കണ്ടു.
- മൂന്ന് റെസീറ്റുകളുടെ ഹാഷ്-ചെയിൻ ചെയിൻ കെട്ടിപ്പൊന്നു.
- ചെയിൻ മധ്യഭാഗം തട്ടിയെടുത്ത് ഒപ്പം ഒപ്പു പരാജയം, ചെയിൻ-ലിങ്ക് പരാജയം ഉണ്ടായി.
- ടൂൾ ഫംഗ്ഷൻ സ്വയം റെസീറ്റ് ഒപ്പിട്ടുകൊണ്ട് വൃത്തിയാക്കി.

**വിപുലീകരണ ചലഞ്ച്.** റെസീറ്റ് സ്കീമയിൽ `request_id` ഫീൽഡ് (വിതരണമായ ട്രേയ്സിംഗിനുള്ള UUID) ചേർക്കുക. `make_receipt` അപ്ഡേറ്റ് ചെയ്ത് ഇത് ഉൾപ്പെടുത്തുക, ശേഷം റെസീറുകൾ എല്ലാം തലക്കെട്ട് മുതല്‍ ശരിയായി പരിശോധിക്കപ്പെടുന്നു എന്ന് ഉറപ്പാക്കുക. പിന്നെ ഒപ്പിട്ട ശേഷം ഫീൽഡ് മാറ്റി പരിശോധന പരാജയമായെന്ന് സ്ഥിരീകരിക്കുക. ഇതിലൂടെ ഓരോ ബൈറ്റും ഒപ്പിൽ എങ്ങനെ കാനോണിക്കൽ എൻকോടിങ്ങിൽ സംഭാവന ചെയ്യുന്നുവെന്ന് മനസ്സിലാക്കേണ്ടി വരും.

**പ്രധാനമായ കിഴവ്.** റെസീറുകൾ മൂന്ന് കാര്യങ്ങൾ മാത്രം തെളിയിക്കുന്നു: ഏടിട്ടു നൽകൽ (ഈ കീ ഈ ഉള്ളടക്കം ഒപ്പിട്ടു), അഖണ്ഡത (ഒപ്പിട്ടതിനുശേഷം ഉള്ളടക്കം മാറ്റപ്പെട്ടിട്ടില്ല), ക്രമീകരണം (ഈ റെസീറ്റ് ആ റെസീട്ടിനു ശേഷം വന്നതാണ്). അവ ഏജന്റിന്റെ പ്രവർത്തനം ശരിയാണെന്ന്, `policy_id`-ൽ പേരു പറയപ്പെട്ട നയം ശരിക്കും വിശദീകരിച്ചതാണെന്ന്, അല്ലെങ്കിൽ ഏജന്റ് എല്ലാ നിയമവും പാലിച്ചതാണെന്ന് തെളിയിക്കില്ല. റെസീറുകൾ ഒരു അടിസ്ഥാനമാണ്. ഭരണകോശം നിങ്ങൾ കെട്ടിടത്തിലെ സിസ്റ്റമാണ്.

അന്ന് അതിലെ അതിതിമിതിയെ മനസ്സിലാക്കി പാഠപുസ്തകം README വീണ്ടും വായിക്കുക. ടീമുകൾ മിക്കവാറും തെറ്റിക്കുന്നതും "ഞങ്ങൾക്ക് റെസീറുണ്ട്" എന്ന് "ഞങ്ങൾ ശാസിച്ചിരിക്കുന്നു" എന്ന് കരുതുകയാണ്. അതില്ല. റെസീറുകൾ ഏജന്റ് രീതിശാസ്ത്രം ഓഡിറ്റബിൾ ആക്കുന്നു. അത് ശരിയാക്കാറില്ല.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**അറിയിപ്പ്**:
ഈ രേഖ AI പരിഭാഷാ സേവനം [Co-op Translator](https://github.com/Azure/co-op-translator) ഉപയോഗിച്ച് പരിഭാഷപ്പെടുത്തിയതാണ്. ഞങ്ങൾ കൃത്യതയ്ക്കായി ശ്രമിക്കുന്നുവെങ്കിലും, ഓട്ടോമേറ്റഡ് പരിഭാഷകളിൽ പിഴവുകൾ അല്ലെങ്കിൽ തെറ്റായ വിവരങ്ങൾ ഉണ്ടാകാൻ സാധ്യതയുണ്ട്. അതിന്റെ സ്വാഭാവിക ഭാഷയിലുള്ള അസൽ രേഖയാണ് പ്രാമാണികമായ ഉറവിടമായി പരിഗണിക്കേണ്ടത്. നിർണായകമായ വിവരങ്ങൾക്ക്, പ്രൊഫഷണൽ മനുഷ്യ പരിഭാഷ ശുപാർശ ചെയ്യുന്നു. ഈ പരിഭാഷ ഉപയോഗിച്ച് ഉണ്ടാകുന്ന തെറ്റിദ്ധാരണകൾ അല്ലെങ്കിൽ തെറ്റായ വ്യാഖ്യാനങ്ങൾക്കായി ഞങ്ങൾ ഉത്തരവാദികളല്ല.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
